# Notebook 10. Generador de datos sintéticos

## 1. Validación de parámetros

In [0]:

'''from datetime import datetime, date, timezone
from numbers import Integral, Real


def generar_ais_sintetico(
    cantidad_registros=10000,
    porcentaje_invalidos=20,
    fecha_inicio="2026-11-23",
    fecha_fin="2026-11-30",
    semilla=42,
    guardar_archivo=False
):

    # 1. Validar cantidad de registros
    if (
        isinstance(cantidad_registros, bool)
        or not isinstance(cantidad_registros, Integral)
        or cantidad_registros <= 0
    ):
        raise ValueError(
            "cantidad_registros debe ser un entero positivo."
        )

    # 2. Validar porcentaje de registros inválidos
    if (
        isinstance(porcentaje_invalidos, bool)
        or not isinstance(porcentaje_invalidos, Real)
        or not 0 <= porcentaje_invalidos <= 100
    ):
        raise ValueError(
            "porcentaje_invalidos debe estar entre 0 y 100."
        )

    # 3. Validar fechas
    try:
        inicio = date.fromisoformat(fecha_inicio)
        fin = date.fromisoformat(fecha_fin)
    except (ValueError, TypeError):
        raise ValueError(
            "Las fechas deben tener formato YYYY-MM-DD."
        )

    if fin < inicio:
        raise ValueError(
            "fecha_fin no puede ser anterior a fecha_inicio."
        )

    # 4. Validar semilla
    if isinstance(semilla, bool) or not isinstance(semilla, Integral):
        raise ValueError(
            "semilla debe ser un número entero."
        )

    # 5. Validar opción de almacenamiento
    if not isinstance(guardar_archivo, bool):
        raise ValueError(
            "guardar_archivo debe ser True o False."
        )

    # 6. Calcular distribución esperada
    cantidad_invalidos = round(
        cantidad_registros * porcentaje_invalidos / 100
    )

    cantidad_validos = (
        cantidad_registros - cantidad_invalidos
    )

    # 7. Generar nombre automático
    fecha_generacion = datetime.now(timezone.utc)

    nombre_archivo = (
        f"AIS_SINTETICO_"
        f"{fecha_generacion:%Y%m%d_%H%M%S_%f}.csv"
    )

    # 8. Mostrar configuración
    print("CONFIGURACIÓN DE GENERACIÓN AIS")
    print("=" * 60)

    print(f"Registros totales: {cantidad_registros:,}")
    print(f"Registros válidos: {cantidad_validos:,}")
    print(f"Registros inválidos: {cantidad_invalidos:,}")
    print(f"Porcentaje inválidos: {porcentaje_invalidos}%")
    print(f"Fecha inicial: {inicio}")
    print(f"Fecha final: {fin}")
    print(f"Semilla aleatoria: {semilla}")
    print(f"Guardar archivo: {guardar_archivo}")
    print(f"Nombre del archivo: {nombre_archivo}")

    print("\n[OK] Parámetros validados correctamente.")
    print("[INFO] No se generaron ni almacenaron registros.")

    return {
        "cantidad_registros": cantidad_registros,
        "cantidad_validos": cantidad_validos,
        "cantidad_invalidos": cantidad_invalidos,
        "fecha_inicio": inicio,
        "fecha_fin": fin,
        "semilla": semilla,
        "guardar_archivo": guardar_archivo,
        "nombre_archivo": nombre_archivo
    }'''


## 2. Perfilamiento de datos históricos

In [0]:
'''from pyspark.sql import functions as F

TABLA_REFERENCIA = "workspace.oceanwatch.ais_silver"

df_historico = spark.table(TABLA_REFERENCIA)

print("ESQUEMA DE DATOS HISTÓRICOS")
print("=" * 60)
df_historico.printSchema()

print("\nESTADÍSTICAS DE VARIABLES NUMÉRICAS")
print("=" * 60)

display(
    df_historico.select(
        F.min("LAT").alias("lat_min"),
        F.max("LAT").alias("lat_max"),
        F.avg("LAT").alias("lat_promedio"),
        F.min("LON").alias("lon_min"),
        F.max("LON").alias("lon_max"),
        F.avg("LON").alias("lon_promedio"),
        F.min("SOG").alias("sog_min"),
        F.max("SOG").alias("sog_max"),
        F.avg("SOG").alias("sog_promedio"),
        F.approx_count_distinct("MMSI").alias("mmsi_aprox")
    )
)

print("\nTIPOS DE EMBARCACIÓN MÁS FRECUENTES")
print("=" * 60)

display(
    df_historico
    .groupBy("VesselType")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
)'''


ESQUEMA DE DATOS HISTÓRICOS
root
 |-- MMSI: string (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: integer (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: double (nullable = true)
 |-- Width: double (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- origen_datos: string (nullable = true)
 |-- fecha_ingesta: timestamp (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- VesselType_Descripcion: string (nullable = true)
 |-- VesselType_Categoria: string (nullable = true)


ESTADÍSTICAS DE VARIABLES 

lat_min,lat_max,lat_promedio,lon_min,lon_max,lon_promedio,sog_min,sog_max,sog_promedio,mmsi_aprox
0.62368,89.6552,35.43144237819282,-161.65198,146.3172,-93.41922515891035,0.0,40.0,2.3604064183946716,33021



TIPOS DE EMBARCACIÓN MÁS FRECUENTES


VesselType,count
31,16518835
37,14427512
60,4776597
30,4000179
36,3862501
70,3836251
90,3835366
52,1914504
80,1788816
57,1323668


## 3. Muestreo de datos históricos

In [0]:
'''from pyspark.sql import functions as F

TABLA_REFERENCIA = "workspace.oceanwatch.ais_silver"

df_historico = spark.table(TABLA_REFERENCIA)

# Seleccionar una muestra aleatoria reproducible
df_muestra = (
    df_historico
    .orderBy(F.rand(seed=42))
    .limit(10000)
    .cache()
)

# Materializar la muestra
total_muestra = df_muestra.count()

print("VALIDACIÓN DE MUESTRA HISTÓRICA")
print("=" * 60)
print(f"Registros seleccionados: {total_muestra:,}")

print("\nDISTRIBUCIÓN DE TIPOS DE EMBARCACIÓN")
print("=" * 60)

display(
    df_muestra
    .groupBy("VesselType")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

print("\nVISTA PREVIA DE LA MUESTRA")
print("=" * 60)

display(df_muestra.limit(5))'''


VALIDACIÓN DE MUESTRA HISTÓRICA
Registros seleccionados: 10,000

DISTRIBUCIÓN DE TIPOS DE EMBARCACIÓN


VesselType,count
31,2745
37,2419
60,810
90,658
30,642
36,615
70,596
52,337
80,312
57,206



VISTA PREVIA DE LA MUESTRA


MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,fecha,origen_datos,fecha_ingesta,archivo_origen,VesselType_Descripcion,VesselType_Categoria
367396520,2023-06-06T21:46:04.000Z,36.96622,-76.06404,2.1,286.7,null,CROSBY RAM,IMO9558115,WDH3316,31,0,28.0,9.0,3.6,null,A,2023-06-06,NOAA_AIS,2026-11-23T02:18:52.470Z,/Volumes/workspace/oceanwatch/ais_landing/incoming/AIS_2023_06_06.csv,Towing,Tug & tow
367477690,2023-06-06T11:03:53.000Z,34.71778,-76.69567,0.0,182.4,null,FORT MACON,IMO5238779,WDF6773,31,0,26.0,7.0,3.1,52,A,2023-06-06,NOAA_AIS,2026-11-23T02:18:52.470Z,/Volumes/workspace/oceanwatch/ais_landing/incoming/AIS_2023_06_06.csv,Towing,Tug & tow
316050841,2023-06-03T10:05:54.000Z,49.31458,-123.13672,6.4,301.0,null,OCEAN CYPRESS,IMO9896490,null,52,5,28.0,13.0,6.2,52,A,2023-06-03,NOAA_AIS,2026-11-23T02:18:52.470Z,/Volumes/workspace/oceanwatch/ais_landing/incoming/AIS_2023_06_03.csv,Tug,Tug & tow
303668000,2023-06-05T19:47:13.000Z,48.16791,-125.30378,10.8,209.3,null,GULF RELIANCE,IMO9398474,WDD2703,31,0,41.0,12.0,null,52,A,2023-06-05,NOAA_AIS,2026-11-23T02:18:52.470Z,/Volumes/workspace/oceanwatch/ais_landing/incoming/AIS_2023_06_05.csv,Towing,Tug & tow
566644000,2023-06-07T15:42:52.000Z,27.83305,-97.50148,0.0,160.0,null,EAGLE SAN JUAN,IMO9594846,9V9333,80,5,274.0,49.0,17.2,80,A,2023-06-07,NOAA_AIS,2026-11-23T02:18:52.470Z,/Volumes/workspace/oceanwatch/ais_landing/incoming/AIS_2023_06_07.csv,Tanker,Tanker


## 4. Generación de registros nuevos AIS (Base)

In [0]:
'''from datetime import date

configuracion = {
    "cantidad_registros": 10000,
    "porcentaje_invalidos": 20,
    "fecha_inicio": date(2023, 6, 8),
    "fecha_fin": date(2023, 6, 10),
    "semilla": 42,
    "guardar_archivo": False
}

print("CONFIGURACIÓN DEFINIDA CORRECTAMENTE")
print("=" * 60)

for parametro, valor in configuracion.items():
    print(f"{parametro}: {valor}")'''


CONFIGURACIÓN DEFINIDA CORRECTAMENTE
cantidad_registros: 10000
porcentaje_invalidos: 20
fecha_inicio: 2023-06-08
fecha_fin: 2023-06-10
semilla: 42
guardar_archivo: False


In [0]:
'''from pyspark.sql import functions as F

columnas_ais = [
    "MMSI", "BaseDateTime", "LAT", "LON", "SOG",
    "COG", "Heading", "VesselName", "IMO",
    "CallSign", "VesselType", "Status",
    "Length", "Width", "Draft", "Cargo",
    "TransceiverClass"
]

def generar_registros_base(df_muestra, configuracion):

    cantidad = configuracion["cantidad_registros"]
    semilla = configuracion["semilla"]

    # Crear exactamente la cantidad solicitada
    df_indices = spark.range(cantidad)

    # Asignar aleatoriamente un registro de referencia
    df_referencia = (
        df_muestra
        .select(*columnas_ais)
        .withColumn(
            "_indice_muestra",
            F.row_number().over(
                __import__(
                    "pyspark.sql.window",
                    fromlist=["Window"]
                ).Window.orderBy(F.monotonically_increasing_id())
            ) - 1
        )
    )

    total_referencia = df_referencia.count()

    df_indices = df_indices.withColumn(
        "_indice_muestra",
        F.floor(
            F.rand(seed=semilla) * total_referencia
        ).cast("long")
    )

    df_sintetico = (
        df_indices
        .join(df_referencia, on="_indice_muestra")
        .drop("_indice_muestra")
    )

    # Definir intervalo de fechas
    inicio = configuracion["fecha_inicio"]
    fin = configuracion["fecha_fin"]

    inicio_ts = int(
        __import__("datetime").datetime.combine(
            inicio,
            __import__("datetime").time.min,
            tzinfo=__import__("datetime").timezone.utc
        ).timestamp()
    )

    fin_ts = int(
        __import__("datetime").datetime.combine(
            fin,
            __import__("datetime").time.max,
            tzinfo=__import__("datetime").timezone.utc
        ).timestamp()
    )

    # Generar fechas aleatorias dentro del intervalo
    df_sintetico = df_sintetico.withColumn(
        "BaseDateTime",
        F.from_unixtime(
            F.floor(
                F.lit(inicio_ts) +
                F.rand(seed=semilla + 1) *
                (fin_ts - inicio_ts + 1)
            ).cast("long"),
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )

    return df_sintetico.select(*columnas_ais)


# Generación de prueba
df_sintetico_base = generar_registros_base(
    df_muestra,
    configuracion
)

print("REGISTROS SINTÉTICOS BASE")
print("=" * 60)

print(f"Total: {df_sintetico_base.count():,}")

display(
    df_sintetico_base.select(
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON",
        "SOG",
        "VesselType"
    ).limit(10)
)'''


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


REGISTROS SINTÉTICOS BASE
Total: 10,000


MMSI,BaseDateTime,LAT,LON,SOG,VesselType
367396520,2023-06-09T16:13:46,36.96622,-76.06404,2.1,31
367477690,2023-06-09T23:01:13,34.71778,-76.69567,0.0,31
316050841,2023-06-10T02:10:13,49.31458,-123.13672,6.4,52
566644000,2023-06-08T20:28:36,27.83305,-97.50148,0.0,80
367029690,2023-06-10T16:23:33,39.19864,-76.5282,0.1,31
368005270,2023-06-09T00:24:23,47.26348,-122.63847,0.0,37
241330000,2023-06-08T10:00:08,28.38526,-120.91191,10.1,80
338333468,2023-06-08T16:12:15,40.85423,-73.64628,0.1,36
368160240,2023-06-10T16:26:12,27.1842,-94.31545,3.8,0
368280920,2023-06-10T23:04:16,29.58964,-94.93856,7.1,0


## 5. Validación de Registros sintéticos base

In [0]:
'''from pyspark.sql import functions as F

df_validacion = df_sintetico_base.cache()

total = df_validacion.count()

resumen = df_validacion.agg(
    F.min("BaseDateTime").alias("fecha_minima"),
    F.max("BaseDateTime").alias("fecha_maxima"),
    F.countDistinct("VesselType").alias("tipos_embarcacion"),
    F.sum(F.col("MMSI").isNull().cast("int")).alias("mmsi_nulos")
).first()

fecha_inicio = configuracion["fecha_inicio"].isoformat()
fecha_fin = configuracion["fecha_fin"].isoformat()

fechas_fuera_rango = df_validacion.filter(
    (F.substring("BaseDateTime", 1, 10) < fecha_inicio) |
    (F.substring("BaseDateTime", 1, 10) > fecha_fin)
).count()

print("VALIDACIÓN DE REGISTROS AIS SINTÉTICOS")
print("=" * 60)

print(f"Registros generados: {total:,}")
print(f"Registros esperados: {configuracion['cantidad_registros']:,}")
print(f"Columnas generadas: {len(df_validacion.columns)}")
print(f"Columnas esperadas: {len(columnas_ais)}")
print(f"Fecha mínima: {resumen['fecha_minima']}")
print(f"Fecha máxima: {resumen['fecha_maxima']}")
print(f"Fechas fuera del rango: {fechas_fuera_rango}")
print(f"Tipos de embarcación: {resumen['tipos_embarcacion']}")
print(f"MMSI nulos: {resumen['mmsi_nulos']}")

print("\nRESULTADO")
print("=" * 60)

if (
    total == configuracion["cantidad_registros"]
    and df_validacion.columns == columnas_ais
    and fechas_fuera_rango == 0
):
    print("[OK] Cantidad, estructura y fechas correctas")
else:
    print("[REVISAR] Se identificaron diferencias")'''


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VALIDACIÓN DE REGISTROS AIS SINTÉTICOS
Registros generados: 10,000
Registros esperados: 10,000
Columnas generadas: 17
Columnas esperadas: 17
Fecha mínima: 2023-06-08T00:00:03
Fecha máxima: 2023-06-10T23:59:59
Fechas fuera del rango: 0
Tipos de embarcación: 49
MMSI nulos: 0

RESULTADO
[OK] Cantidad, estructura y fechas correctas


## 6. Validación de introducción de errores de calidad

In [0]:

'''from pyspark.sql import functions as F
from pyspark.sql.window import Window

cantidad_invalidos = round(
    configuracion["cantidad_registros"]
    * configuracion["porcentaje_invalidos"] / 100
)

# Numerar los registros de forma reproducible
df_numerado = (
    df_sintetico_base
    .withColumn(
        "_orden",
        F.row_number().over(
            Window.orderBy(
                F.rand(seed=configuracion["semilla"] + 2)
            )
        )
    )
)

# Distribuir los registros inválidos entre cinco reglas
df_numerado = df_numerado.withColumn(
    "_regla_error",
    F.when(
        F.col("_orden") <= cantidad_invalidos,
        F.pmod(F.col("_orden") - 1, F.lit(5)) + 1
    ).otherwise(F.lit(0))
)

# Aplicar errores controlados
df_sintetico_final = (
    df_numerado
    .withColumn(
        "MMSI",
        F.when(F.col("_regla_error") == 1, F.lit("ABC123"))
         .otherwise(F.col("MMSI"))
    )
    .withColumn(
        "BaseDateTime",
        F.when(F.col("_regla_error") == 2, F.lit("FECHA_INVALIDA"))
         .otherwise(F.col("BaseDateTime"))
    )
    .withColumn(
        "LAT",
        F.when(F.col("_regla_error") == 3, F.lit("95.0"))
         .otherwise(F.col("LAT").cast("string"))
    )
    .withColumn(
        "LON",
        F.when(F.col("_regla_error") == 4, F.lit("-190.0"))
         .otherwise(F.col("LON").cast("string"))
    )
    .withColumn(
        "SOG",
        F.when(F.col("_regla_error") == 5, F.lit("55.0"))
         .otherwise(F.col("SOG").cast("string"))
    )
    .select(*columnas_ais, "_regla_error")
    .cache()
)

# Materializar el resultado
total_final = df_sintetico_final.count()

print("GENERACIÓN DE ERRORES AIS")
print("=" * 60)
print(f"Total de registros: {total_final:,}")
print(f"Registros inválidos esperados: {cantidad_invalidos:,}")

display(
    df_sintetico_final
    .groupBy("_regla_error")
    .count()
    .orderBy("_regla_error")
)'''


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


GENERACIÓN DE ERRORES AIS
Total de registros: 10,000
Registros inválidos esperados: 2,000


_regla_error,count
0,8000
1,400
2,400
3,400
4,400
5,400


## 7. Validación de calidad de datos sintéticos

In [0]:
'''from pyspark.sql import functions as F

# Reemplazar la validación de fechas
df_control = df_control.withColumn(
    "RGN_S03",
    F.try_to_timestamp(
        F.col("BaseDateTime"),
        F.lit("yyyy-MM-dd'T'HH:mm:ss")
    ).isNotNull()
)

# Recalcular el resultado general de calidad
df_control = df_control.withColumn(
    "registro_valido",
    F.coalesce(
        F.col("RGN_S02") &
        F.col("RGN_S03") &
        F.col("RGN_S04") &
        F.col("RGN_S05") &
        F.col("RGN_S06"),
        F.lit(False)
    )
)

print("VALIDACIÓN GENERAL")
print("=" * 60)

display(
    df_control
    .groupBy("registro_valido")
    .count()
)

print("\nDISTRIBUCIÓN DE ERRORES")
print("=" * 60)

display(
    df_control
    .groupBy("_regla_error")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            (~F.col("registro_valido")).cast("int")
        ).alias("registros_invalidos")
    )
    .orderBy("_regla_error")
)

print("\nINCUMPLIMIENTOS POR REGLA")
print("=" * 60)

display(
    df_control.agg(
        *[
            F.sum(
                (~F.coalesce(F.col(regla), F.lit(False)))
                .cast("int")
            ).alias(regla)
            for regla in [
                "RGN_S02", "RGN_S03", "RGN_S04",
                "RGN_S05", "RGN_S06"
            ]
        ]
    )
)'''


VALIDACIÓN GENERAL


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


registro_valido,count
false,2000
true,8000



DISTRIBUCIÓN DE ERRORES


_regla_error,registros,registros_invalidos
0,8000,0
1,400,400
2,400,400
3,400,400
4,400,400
5,400,400



INCUMPLIMIENTOS POR REGLA


RGN_S02,RGN_S03,RGN_S04,RGN_S05,RGN_S06
400,400,400,400,400


## 8. Generación de datos Sintéticos

In [0]:
from datetime import datetime, date, timedelta, timezone
from numbers import Integral, Real
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType
)
import math
import random
import uuid
import re


def generar_ais_sintetico(
    cantidad_registros=10000,
    porcentaje_invalidos=20,
    fecha_inicio="2023-06-08",
    fecha_fin="2023-06-10",
    semilla=42,
    guardar_archivo=False
):
    """
    Genera registros AIS sintéticos utilizando datos históricos
    de Silver e introduce errores controlados.

    guardar_archivo=False:
        Genera y valida los registros sin escribir archivos.

    guardar_archivo=True:
        Genera un CSV y lo publica en Landing, lo que puede
        activar el Job incremental de OceanWatch.
    """

    # ========================================================
    # 1. VALIDACIÓN DE PARÁMETROS
    # ========================================================

    if (
        isinstance(cantidad_registros, bool)
        or not isinstance(cantidad_registros, Integral)
        or cantidad_registros <= 0
    ):
        raise ValueError(
            "cantidad_registros debe ser un entero positivo"
        )

    if (
        isinstance(porcentaje_invalidos, bool)
        or not isinstance(porcentaje_invalidos, Real)
        or not math.isfinite(porcentaje_invalidos)
        or not 0 <= porcentaje_invalidos <= 100
    ):
        raise ValueError(
            "porcentaje_invalidos debe estar entre 0 y 100"
        )

    if (
        isinstance(semilla, bool)
        or not isinstance(semilla, Integral)
    ):
        raise ValueError("semilla debe ser un número entero")

    if not isinstance(guardar_archivo, bool):
        raise ValueError(
            "guardar_archivo debe ser True o False"
        )

    def convertir_fecha(valor):
        if isinstance(valor, datetime):
            return valor.date()
        if isinstance(valor, date):
            return valor
        if isinstance(valor, str):
            return date.fromisoformat(valor)
        raise ValueError("Formato de fecha no válido")

    inicio = convertir_fecha(fecha_inicio)
    fin = convertir_fecha(fecha_fin)

    if inicio > fin:
        raise ValueError(
            "fecha_inicio no puede superar fecha_fin"
        )

    cantidad_invalidos = round(
        cantidad_registros * porcentaje_invalidos / 100
    )

    cantidad_validos = (
        cantidad_registros - cantidad_invalidos
    )

    columnas_ais = [
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON",
        "SOG",
        "COG",
        "Heading",
        "VesselName",
        "IMO",
        "CallSign",
        "VesselType",
        "Status",
        "Length",
        "Width",
        "Draft",
        "Cargo",
        "TransceiverClass"
    ]

    print("CONFIGURACIÓN DE GENERACIÓN")
    print("=" * 65)
    print(f"Registros solicitados: {cantidad_registros:,}")
    print(f"Válidos esperados: {cantidad_validos:,}")
    print(f"Inválidos esperados: {cantidad_invalidos:,}")
    print(f"Período: {inicio} a {fin}")
    print(f"Semilla: {semilla}")
    print(f"Guardar archivo: {guardar_archivo}")

    # ========================================================
    # 2. OBTENCIÓN DE REFERENCIAS HISTÓRICAS
    # ========================================================

    tabla_historica = (
        "workspace.oceanwatch.ais_silver"
    )

    df_historico = (
        spark.table(tabla_historica)
        .select(*columnas_ais)
    )

    # Muestreo reproducible sin ordenar la tabla completa.
    # La muestra está limitada para evitar sobrecargar
    # la memoria del driver.

    df_muestra = (
        df_historico
        .sample(
            withReplacement=False,
            fraction=0.001,
            seed=semilla
        )
        .limit(20000)
    )

    referencias = df_muestra.collect()

    if not referencias:
        raise ValueError(
            "No se obtuvieron referencias históricas"
        )

    # ========================================================
    # 3. FILTRADO DE REFERENCIAS VÁLIDAS
    # ========================================================

    referencias_validas = []

    for fila in referencias:

        registro = fila.asDict()

        try:
            mmsi = str(registro["MMSI"])

            if not re.fullmatch(r"[0-9]{9}", mmsi):
                continue

            lat = float(registro["LAT"])
            lon = float(registro["LON"])
            sog = float(registro["SOG"])

            if not (
                math.isfinite(lat)
                and math.isfinite(lon)
                and math.isfinite(sog)
                and -90 <= lat <= 90
                and -180 <= lon <= 180
                and 0 <= sog <= 40
            ):
                continue

            referencias_validas.append(registro)

        except (ValueError, TypeError):
            continue

    if not referencias_validas:
        raise ValueError(
            "No se encontraron referencias históricas válidas"
        )

    print("\nREFERENCIAS HISTÓRICAS")
    print("=" * 65)
    print(
        f"Referencias obtenidas: {len(referencias):,}"
    )
    print(
        f"Referencias válidas: {len(referencias_validas):,}"
    )

    # ========================================================
    # 4. GENERACIÓN DE REGISTROS SINTÉTICOS
    # ========================================================

    rng = random.Random(semilla)

    inicio_datetime = datetime.combine(
        inicio,
        datetime.min.time()
    )

    segundos_intervalo = (
        (fin - inicio).days + 1
    ) * 86400

    registros = []

    for i in range(cantidad_registros):

        referencia = rng.choice(
            referencias_validas
        ).copy()

        segundos = rng.randrange(
            segundos_intervalo
        )

        fecha_generada = (
            inicio_datetime
            + timedelta(seconds=segundos)
        )

        referencia["BaseDateTime"] = (
            fecha_generada.strftime(
                "%Y-%m-%dT%H:%M:%S"
            )
        )

        registros.append(referencia)

    # ========================================================
    # 5. INTRODUCCIÓN DE ERRORES CONTROLADOS
    # ========================================================

    indices_invalidos = rng.sample(
        range(cantidad_registros),
        cantidad_invalidos
    )

    errores_por_regla = {
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        5: 0
    }

    for posicion, indice in enumerate(
        indices_invalidos
    ):

        regla = (posicion % 5) + 1

        registro = registros[indice]

        if regla == 1:
            # RGN_S02: MMSI inválido
            registro["MMSI"] = "ABC123"

        elif regla == 2:
            # RGN_S03: Fecha inválida
            registro["BaseDateTime"] = (
                "FECHA_INVALIDA"
            )

        elif regla == 3:
            # RGN_S04: Latitud inválida
            registro["LAT"] = "95.0"

        elif regla == 4:
            # RGN_S05: Longitud inválida
            registro["LON"] = "-190.0"

        elif regla == 5:
            # RGN_S06: Velocidad inválida
            registro["SOG"] = "55.0"

        errores_por_regla[regla] += 1

    # ========================================================
    # 6. CONSTRUCCIÓN DEL DATAFRAME
    # ========================================================

    esquema = StructType([
        StructField(
            columna,
            StringType(),
            True
        )
        for columna in columnas_ais
    ])

    datos_salida = [
        tuple(
            None
            if registro.get(columna) is None
            else str(registro.get(columna))
            for columna in columnas_ais
        )
        for registro in registros
    ]

    df_resultado = spark.createDataFrame(
        datos_salida,
        schema=esquema
    ).cache()

    # ========================================================
    # 7. VALIDACIÓN DE REGLAS DE CALIDAD
    # ========================================================

    df_validacion = (
        df_resultado

        # RGN_S02: MMSI de nueve dígitos
        .withColumn(
            "_r02",
            F.coalesce(
                F.col("MMSI").rlike(
                    r"^[0-9]{9}$"
                ),
                F.lit(False)
            )
        )

        # RGN_S03: Fecha interpretable
        .withColumn(
            "_r03",
            F.try_to_timestamp(
                F.col("BaseDateTime"),
                F.lit(
                    "yyyy-MM-dd'T'HH:mm:ss"
                )
            ).isNotNull()
        )

        # RGN_S04: Latitud válida
        .withColumn(
            "_r04",
            F.coalesce(
                F.expr(
                    "try_cast(LAT AS DOUBLE)"
                ).between(-90, 90),
                F.lit(False)
            )
        )

        # RGN_S05: Longitud válida
        .withColumn(
            "_r05",
            F.coalesce(
                F.expr(
                    "try_cast(LON AS DOUBLE)"
                ).between(-180, 180),
                F.lit(False)
            )
        )

        # RGN_S06: Velocidad válida
        .withColumn(
            "_r06",
            F.coalesce(
                F.expr(
                    "try_cast(SOG AS DOUBLE)"
                ).between(0, 40),
                F.lit(False)
            )
        )
    )

    df_validacion = df_validacion.withColumn(
        "_registro_valido",
        F.col("_r02")
        & F.col("_r03")
        & F.col("_r04")
        & F.col("_r05")
        & F.col("_r06")
    )

    resumen = df_validacion.agg(
        F.count("*").alias("total"),
        F.sum(
            F.col("_registro_valido").cast("int")
        ).alias("validos"),
        *[
            F.sum(
                (~F.col(regla)).cast("int")
            ).alias(regla)
            for regla in [
                "_r02", "_r03", "_r04",
                "_r05", "_r06"
            ]
        ]
    ).first()

    total = resumen["total"]
    validos = resumen["validos"]
    invalidos = total - validos

    print("\nVALIDACIÓN DE CALIDAD")
    print("=" * 65)
    print(f"Total: {total:,}")
    print(f"Válidos: {validos:,}")
    print(f"Inválidos: {invalidos:,}")

    print("\nINCUMPLIMIENTOS POR REGLA")
    print("=" * 65)

    for numero in range(2, 7):
        campo = f"_r{numero:02d}"
        codigo = f"RGN_S{numero:02d}"
        print(
            f"{codigo}: {resumen[campo]:,}"
        )

    # Validar también la distribución exacta por regla.
    cantidades_esperadas = {
        f"_r{numero:02d}": errores_por_regla[numero - 1]
        for numero in range(2, 7)
    }

    distribucion_correcta = all(
        resumen[campo] == esperado
        for campo, esperado in cantidades_esperadas.items()
    )

    if not (
        total == cantidad_registros
        and validos == cantidad_validos
        and invalidos == cantidad_invalidos
        and distribucion_correcta
    ):
        raise ValueError(
            "La validación de calidad no coincide "
            "con la distribución solicitada"
        )

    print("\n[OK] Validación completada correctamente")

    # ========================================================
    # 8. GENERACIÓN Y PUBLICACIÓN OPCIONAL DEL CSV
    # ========================================================

    if guardar_archivo:

        nombre_archivo = (
            "AIS_SINTETICO_"
            + datetime.now(
                timezone.utc
            ).strftime(
                "%Y%m%d_%H%M%S_%f"
            )
            + "_"
            + uuid.uuid4().hex[:8]
            + ".csv"
        )

        ruta_landing = (
            "/Volumes/workspace/oceanwatch/"
            "ais_landing/incoming/"
        )

        ruta_temporal = (
            "/Volumes/workspace/oceanwatch/"
            "ais_landing/"
            "_temporal_sintetico_"
            + uuid.uuid4().hex
        )

        print("\nGENERACIÓN DEL ARCHIVO CSV")
        print("=" * 65)

        # Escribir fuera de la carpeta monitoreada.
        (
            df_resultado
            .coalesce(1)
            .write
            .format("csv")
            .mode("overwrite")
            .option("header", "true")
            .option("delimiter", ",")
            .option("nullValue", "")
            .save(ruta_temporal)
        )

        # Identificar el único archivo CSV de Spark.
        archivos_csv = [
            archivo.path
            for archivo in dbutils.fs.ls(ruta_temporal)
            if archivo.name.startswith("part-")
            and archivo.name.endswith(".csv")
        ]

        if len(archivos_csv) != 1:
            raise RuntimeError(
                "Se esperaba un único archivo CSV temporal"
            )

        ruta_final = ruta_landing + nombre_archivo

        # Publicación del archivo en Landing.
        # Puede activar el Job File arrival.
        dbutils.fs.mv(
            archivos_csv[0],
            ruta_final
        )

        dbutils.fs.rm(
            ruta_temporal,
            recurse=True
        )

        print("[OK] Archivo CSV publicado")
        print(f"Ruta: {ruta_final}")
        print(
            "[AVISO] El Job incremental puede activarse"
        )

    else:

        print("\nALMACENAMIENTO")
        print("=" * 65)
        print("[OK] No se generó ningún archivo CSV")
        print("[OK] No se publicó nada en Landing")

    return df_resultado



In [0]:
df_ais_generado = generar_ais_sintetico(
    cantidad_registros=10000,
    porcentaje_invalidos=20,
    fecha_inicio="2023-06-08",
    fecha_fin="2023-06-10",
    semilla=42,
    guardar_archivo=True
)


CONFIGURACIÓN DE GENERACIÓN
Registros solicitados: 10,000
Válidos esperados: 8,000
Inválidos esperados: 2,000
Período: 2023-06-08 a 2023-06-10
Semilla: 42
Guardar archivo: True

REFERENCIAS HISTÓRICAS
Referencias obtenidas: 20,000
Referencias válidas: 20,000

VALIDACIÓN DE CALIDAD
Total: 10,000
Válidos: 8,000
Inválidos: 2,000

INCUMPLIMIENTOS POR REGLA
RGN_S02: 400
RGN_S03: 400
RGN_S04: 400
RGN_S05: 400
RGN_S06: 400

[OK] Validación completada correctamente

GENERACIÓN DEL ARCHIVO CSV
[OK] Archivo CSV publicado
Ruta: /Volumes/workspace/oceanwatch/ais_landing/incoming/AIS_SINTETICO_20261123_045341_839912_8ead3bff.csv
[AVISO] El Job incremental puede activarse
